# 02 — Calidad, Limpieza y Preparación

Cada decisión de preparación incluye: evidencia observada → acción aplicada → impacto en el dataset.
Se registra cada transformación en `logs/pipeline_log.csv`.

In [ ]:
import pandas as pd
import numpy as np
import os

# Cargamos el dataset
df = pd.read_json('streaming_users_dirty.json')
df = df.copy()

N0 = len(df)
log = []  # registro ETL

def auditar(paso, desc):
  log.append({
      'Paso': paso, 'Descripción': desc,
      'Filas': df.shape[0], 'Nulos': df.isnull().sum().sum(),
      'Retención (%)': round(len(df)/N0*100, 2)}) # Fixed: Corrected round function arguments

auditar(0, 'Estado original')
print(f"Dataset original: {len(df)} filas, {int(df.isnull().sum().sum())} nulos") # Fixed: Added .sum() for total nulls

Dataset original: 8160 filas, 753 nulos


## Paso 1 — Eliminación de duplicados

**Evidencia**

Una inspección inicial detectó 160 filas duplicadas en total: 126 duplicadas exactas (filas idénticas en todos sus campos) y 34 adicionales detectadas por duplicación de user_id.

**Acción**

 Ambos grupos se eliminaron, conservando solo la primera ocurrencia.

**Impacto esperado**

El conjunto de datos se redujo de 8160 a 8000 filas. Esto evita recuentos inflados y distribuciones sesgadas durante el análisis, asegurando que cada usuario esté representado solo una vez.

In [ ]:
df = df.drop_duplicates()
df = df.drop_duplicates(subset='user_id', keep='first')
auditar(1, 'Eliminación de duplicados')
print("Filas", len(df), "user_id únicos", df['user_id'].is_unique)


Filas 8000 user_id únicos True


## Paso 2 — Normalización de variables categoricas

**Evidencia**

La inspección manual mostró numerosas variantes para cada variable categórica (p. ej., 'Std', 'estandar' para 'Estándar' en planes de suscripción; 'Brazil', 'COL' para países; 'comedy', 'thriler' para géneros). Estas variaciones incluían diferentes idiomas, errores tipográficos, mayúsculas y minúsculas mezcladas y espacios extra.

**Acción**

Se definió una función limpiar (para eliminar espacios en blanco y convertir a minúsculas). Luego, se crearon diccionarios de mapeo exhaustivos (plan_map_full, country_map_full, genre_map_full) para estandarizar todas las variantes conocidas a valores canónicos en español. Estos mapas se utilizaron luego para corregir las columnas.

**Impacto esperado**

Cada columna categórica ahora tiene un conjunto consistente y reducido de valores únicos (3 para subscription_plan, 7 para country, 7 para favorite_genre), lo que permite una agrupación y comparación fiables. Los valores que no pudieron mapearse se convirtieron en NaN.

In [ ]:
def limpiar(s): return s.astype(str).str.strip().str.lower()

plan_map_full = {
    'Básico': 'Básico', 'Basico': 'Básico', 'basico': 'Básico', 'BASICO': 'Básico', 'Basic': 'Básico',
    'Estándar': 'Estándar', 'Estandar': 'Estándar', 'estandar': 'Estándar', 'STANDARD': 'Estándar', 'Std': 'Estándar', 'Estándar ': 'Estándar',
    'Premium': 'Premium', 'premium': 'Premium', 'PREMIUM': 'Premium', 'Premium ': 'Premium', 'Premiun': 'Premium'
}
plan_map_corrected = {}
for k, v in plan_map_full.items():
    plan_map_corrected[limpiar(pd.Series([k])).iloc[0]] = v
df['subscription_plan'] = limpiar(df['subscription_plan']).map(plan_map_corrected)

country_map_full = {
    'Argentina': 'Argentina', 'ARG': 'Argentina', 'arg': 'Argentina',
    'Brasil': 'Brasil', 'Brazil': 'Brasil', 'brasil': 'Brasil', 'BRA': 'Brasil',
    'Chile': 'Chile', 'chile': 'Chile', 'Chile ': 'Chile', 'CHL': 'Chile',
    'Colombia': 'Colombia', 'colombia': 'Colombia', 'COL': 'Colombia',
    'México': 'México', 'Mexico': 'México', 'mexico': 'México', 'MEX': 'México', 'méxico': 'México',
    'Perú': 'Perú', 'Peru': 'Perú', 'PER': 'Perú',
    'Uruguay': 'Uruguay', 'uruguay': 'Uruguay', 'URY': 'Uruguay'
}
country_map_corrected = {}
for k, v in country_map_full.items():
    country_map_corrected[limpiar(pd.Series([k])).iloc[0]] = v
df['country'] = limpiar(df['country']).map(country_map_corrected)

genre_map_full = {
    'Acción': 'Acción', 'ACCIÓN': 'Acción', 'accion': 'Acción', 'Action': 'Acción',
    'Comedia': 'Comedia', 'COMEDIA': 'Comedia', 'Comedia ': 'Comedia', 'comedy': 'Comedia',
    'Crime': 'Crime', 'CRIME': 'Crime',
    'Documental': 'Documental', 'Documentary': 'Documental',
    'Drama': 'Drama', 'DRAMA': 'Drama',
    'Romance': 'Romance', 'ROMANCE': 'Romance',
    'Thriller': 'Thriller', 'THRILLER': 'Thriller', 'Thriller ': 'Thriller', 'thriler': 'Thriller'
}
genre_map_corrected = {}
for k, v in genre_map_full.items():
    genre_map_corrected[limpiar(pd.Series([k])).iloc[0]] = v
df['favorite_genre'] = limpiar(df['favorite_genre']).map(genre_map_corrected)

auditar(2, 'Normalización de subscription_plan')
print("Planes", sorted(df['subscription_plan'].dropna().unique()))
print("Países", sorted(df['country'].dropna().unique()))
print("Géneros", sorted(df['favorite_genre'].dropna().unique()))
auditar(2, 'Normalización de subscription_plan')

Planes ['Básico', 'Estándar', 'Premium']
Países ['Argentina', 'Brasil', 'Chile', 'Colombia', 'México', 'Perú', 'Uruguay']
Géneros ['Acción', 'Comedia', 'Crime', 'Documental', 'Drama', 'Romance', 'Thriller']


## Paso 3 - Conversión de valores imposibles a nulos

**Evidencia**

age: Se encontraron valores fuera de un rango humano realista (p. ej., <13 o >100 años).
monthly_watch_time_mins: Valores negativos o valores extremadamente altos (p. ej., >= 10000 minutos, lo que implica una visualización continua mucho más allá de un mes normal).
customer_support_tickets: Valores negativos o valores atípicos específicos (99, 150) que estaban lejos de la distribución central.

**Acción**

Los valores identificados como imposibles o altamente improbables se reemplazaron por np.nan.

**Impacto esperado**

Se eliminaron puntos de datos poco realistas de age (120 valores), monthly_watch_time_mins (80 valores) y customer_support_tickets (96 valores), preparándolos para una imputación o eliminación adecuada.

In [ ]:
# age fuera de 13 100
m_age = (df['age'] < 13) | (df['age'] > 100)
df.loc[m_age, 'age'] = np.nan

# watch time negativos y valores imposibles
m_wt = (df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] >= 10000)
df.loc[m_wt, 'monthly_watch_time_mins'] = np.nan

# tickets negativos y valores imposibles
m_tk =(df['customer_support_tickets'] < 0) | (df['customer_support_tickets'].isin([99,150]))
df.loc[m_tk, 'customer_support_tickets'] = np.nan
print(f"age imposibles: {m_age.sum()}")
print(f"monthly_watch_time_mins imposibles: {m_wt.sum()}")
print(f"customer_support_tickets imposibles: {m_tk.sum()}")
auditar(3, 'Valores imposibles')


age imposibles: 120
monthly_watch_time_mins imposibles: 80
customer_support_tickets imposibles: 96


## Paso 4 — Parsear y validar fechas

**Evidencia**

La columna last_login_date era una cadena de texto, contenía nulos iniciales (320), cadenas no parseables (reveladas como 769 NaT después de la coerción) y fechas en el futuro (528 valores posteriores al '2026-06-24', lo cual es imposible para un campo de 'último inicio de sesión').

**Acción**

La columna se convirtió a datetime usando pd.to_datetime con errors='coerce' para manejar cadenas no parseables. Las fechas futuras se anularon (→ NaT).

**Impacto esperado**

La columna ahora está correctamente tipada como datetime, y las fechas imposibles se eliminan, dejando 7601 fechas válidas y 399 valores NaT para ser reconocidos explícitamente en análisis posteriores.

In [ ]:
fechas = pd.to_datetime(df['last_login_date'], errors='coerce', format='mixed' , dayfirst=False)
fechas = fechas.where(fechas<= pd.Timestamp('2026-06-24') )
df['last_login_date'] = fechas
print("Fechas valida:", df['last_login_date'].notna().sum())
print("Fechas inválidas:", df['last_login_date'].isna().sum())
auditar(4, 'Validar fechas')

Fechas valida: 7601
Fechas inválidas: 399


## Paso 5 — Tratamiento de `monthly_watch_time_mins` Winsorizar

**Evidencia**

Después de convertir los valores imposibles a NaN, monthly_watch_time_mins aún contenía valores atípicos extremos. Dada su naturaleza, un límite superior es más apropiado que simplemente eliminar filas.

**Acción**

Los valores atípicos se trataron mediante winsorización. Los valores que excedían Q3 + 3 * IQR se limitaron a este límite superior. Este límite se calculó en aproximadamente 2692.55 minutos.

**Impacto esperado**

Se ajustaron 108 valores atípicos extremos, llevando los datos a un rango más realista y representativo, al tiempo que se conservaba la estructura general de los datos.

In [ ]:
Q1, Q3 = df['monthly_watch_time_mins'].quantile([0.25, 0.75])
limite = Q3 + 3.0 * (Q3 - Q1)
n_out = (df['monthly_watch_time_mins'] > limite).sum()
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].clip(upper=limite)
print(f"limite k=3:{limite:1f} / Valores acotados: {n_out}")


limite k=3:2692.550000 / Valores acotados: 108


Diagnostico de mecanismo de faltantes MCAR O MAR



In [ ]:
# Marcamos que filas tienen faltante en cada variable a imputar
df['falta_watch_time'] = df['monthly_watch_time_mins'].isna()
df['falta_age'] = df['age'].isna()

# tasa de faltantes por plan de suscripcion
tasa = pd.DataFrame(df.groupby('subscription_plan')['falta_watch_time'].mean().mul(100).round(2))
tasa['falta_age'] = df.groupby('subscription_plan')['falta_age'].mean().mul(100).round(2)

print(tasa)

# limpiar columnas auxiliares
df = df.drop(columns=['falta_watch_time', 'falta_age'])

                   falta_watch_time  falta_age
subscription_plan                             
Básico                         1.08       1.47
Estándar                       2.27       1.70
Premium                       10.74       1.20


**Diagnóstico del Mecanismo de Datos Faltantes (MCAR o MAR)**

**Evidencia:**

El análisis de la tasa de valores faltantes por plan de suscripción (`tasa` DataFrame generado previamente) muestra lo siguiente:

*   **`monthly_watch_time_mins`:** La tasa de valores faltantes es significativamente más alta para el plan 'Premium' (10.74%) en comparación con 'Básico' (1.08%) y 'Estándar' (2.27%).
*   **`age`:** Las tasas de faltantes son relativamente bajas y similares entre los planes (1.47% para 'Básico', 1.70% para 'Estándar', 1.20% para 'Premium').

**Conclusión:**

Para `monthly_watch_time_mins`, la distribución de los valores faltantes *no es aleatoria* y está relacionada con el `subscription_plan`. Esto sugiere un mecanismo de datos faltantes **Missing At Random (MAR)**. Es decir, la probabilidad de que un valor falte para el tiempo de visualización depende del plan de suscripción, pero no del valor del tiempo de visualización en sí.

Para `age`, dado que las tasas de faltantes son bajas y no muestran una dependencia clara del `subscription_plan`, se podría considerar **Missing Completely At Random (MCAR)**, o al menos un MAR débil en relación con el plan de suscripción.

**Implicación para la imputación:**

El diagnóstico MAR para `monthly_watch_time_mins` justifica la estrategia de imputación utilizada previamente (mediana por grupo de `subscription_plan`), ya que permite que la imputación refleje las diferencias en el patrón de datos faltantes entre los grupos.

## Paso 6 - Imputación de valores faltantes

In [ ]:
# Numericas - Mediana por plan - El comportamiento depende del plan (mecanismo MAR)
for col in ['age', 'monthly_watch_time_mins']:
    df[col] = df.groupby('subscription_plan')[col].transform(lambda x: x.fillna(x.median()))

# Categoricas - Moda global
df['favorite_genre'] = df['favorite_genre'].fillna(df['favorite_genre'].mode()[0])


# last_login_date no se imputa
auditar(6, 'Imputación de valores faltantes')
print(df.isnull().sum())

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins       0
country                       0
favorite_genre                0
last_login_date             399
customer_support_tickets     96
dtype: int64


In [ ]:
# Imputar 'customer_support_tickets' con la mediana global
df['customer_support_tickets'] = df['customer_support_tickets'].fillna(df['customer_support_tickets'].median())
auditar(7, 'Imputación de customer_support_tickets')
print(df.isnull().sum())

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins       0
country                       0
favorite_genre                0
last_login_date             399
customer_support_tickets      0
dtype: int64


**Imputación de `customer_support_tickets` y Decisión sobre `last_login_date`**

**Acción:**

*   **`customer_support_tickets`:** Se imputaron los valores faltantes con la mediana global de la columna. Dada la naturaleza de los tickets de soporte (con la mayoría de los usuarios teniendo 1 o 0 tickets, como se vio en el análisis exploratorio), la mediana global es una imputación robusta y adecuada. No se imputó por `subscription_plan` ya que el diagnóstico de faltantes para variables como `age` (con patrones similares en tasas de faltantes) no mostró una fuerte dependencia del plan.
*   **`last_login_date`:** Como se estableció previamente, esta columna no se imputará. La imputación de fechas de último inicio de sesión es compleja y, sin información adicional o un modelo específico, cualquier imputación podría introducir un sesgo significativo o ser engañosa. Los `NaT` restantes (valores nulos de fecha y hora) se mantendrán para ser tratados o excluidos en análisis específicos que requieran esta columna.

**Impacto esperado:**

Los valores faltantes en `customer_support_tickets` se han resuelto, dejando un conjunto de datos más completo para el análisis. La columna `last_login_date` conserva sus nulos, reflejando la ausencia de datos reales y evitando suposiciones incorrectas.

## Paso 7 - Resumen final

In [ ]:
print(f"Dataset final: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Retención total: {df.shape[0]/8160*100:.1f}%")
print(f"\nNulos finales:\n{df.isnull().sum()}")
print(f"\nPlanes: {df['subscription_plan'].value_counts().to_dict()}")
print(f"\nPaíses: {df['country'].value_counts().to_dict()}")


Dataset final: 8000 filas × 8 columnas
Retención total: 98.0%

Nulos finales:
user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins       0
country                       0
favorite_genre                0
last_login_date             399
customer_support_tickets      0
dtype: int64

Planes: {'Básico': 3600, 'Estándar': 2817, 'Premium': 1583}

Países: {'Chile': 1164, 'México': 1156, 'Brasil': 1156, 'Uruguay': 1143, 'Colombia': 1142, 'Perú': 1134, 'Argentina': 1105}


## Guardado del dataset procesado y log ETL

In [ ]:
os.makedirs('../logs', exist_ok=True)
log_df = pd.DataFrame(log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)
print("Log guardado en logs/pipleline.csv")
print(log_df.to_string(index=False))

Log guardado en logs/pipleline.csv
 Paso                            Descripción  Filas  Nulos  Retención (%)
    0                        Estado original   8160    753         100.00
    1              Eliminación de duplicados   8000    753          98.04
    2     Normalización de subscription_plan   8000    786          98.04
    2     Normalización de subscription_plan   8000    786          98.04
    3                     Valores imposibles   8000   1082          98.04
    4                         Validar fechas   8000   1161          98.04
    6        Imputación de valores faltantes   8000    495          98.04
    7 Imputación de customer_support_tickets   8000    399          98.04


In [ ]:
# Guardar dataset procesado
df.to_csv('/content/streaming_users_clean.csv', index=False)
print("Dataset procesado guardado.")

# Guardar log ETL
import csv

# Asegurarse de que el directorio exista
os.makedirs('../logs', exist_ok=True)

with open('../logs/pipeline_log.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['Paso','Descripción','Filas','Nulos','Retención (%)'])
    writer.writeheader()
    writer.writerows(log)
print("Log ETL guardado.")

Dataset procesado guardado.
Log ETL guardado.
